# 🧠 Tuần 3: Huấn luyện ViT-LoRA + Box Regression Head trên BraTS2020

**Tác giả:** Lâm Gia Bảo  
**Nhiệm vụ:** Huấn luyện ViT-LoRA + Box Regression Head trên tập lát cắt đa phương thức Dataset 4 (BraTS2020).  
Thực hiện thí nghiệm đối sánh hiệu quả phát hiện và đóng khung khối u giữa ảnh tổ hợp 3 kênh [FLAIR, T1ce, T2] so với 1 kênh T1ce đơn lẻ.

**Đặc điểm:**
- 1000 epochs tối đa + Early Stopping (patience=50)
- AdamW optimizer + CosineAnnealingWarmRestarts scheduler (backpropagation with LR scheduling)
- Lưu `loss_best.pth` (val loss thấp nhất) và `loss_latest.pth` (epoch cuối) lên Google Drive
- Hỗ trợ resume từ checkpoint khi Colab hết thời gian
- Mount Google Drive tự động

## Cell 0: Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === CẤU HÌNH ĐƯỜNG DẪN ===
# Thư mục lưu checkpoint & model trên Drive CÁ NHÂN của bạn
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/BrainTumorDetect/checkpoints'

# Thư mục Google Drive CHUNG CỦA NHÓM (đã share) - đẩy model cuối cùng vào đây
# Link: https://drive.google.com/drive/folders/1JUGAxnRcgTf4INQstlUt9YqggO_cbWZ6
# Mỗi thành viên tạo thư mục con riêng bên trong, ví dụ: LamGiaBao/
DRIVE_TEAM_DIR = '/content/drive/MyDrive/BrainTumorDetect/team_shared'

os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_TEAM_DIR, exist_ok=True)

print(f'✅ Checkpoint dir: {DRIVE_CHECKPOINT_DIR}')
print(f'✅ Team dir: {DRIVE_TEAM_DIR}')

In [ ]:
# Clone repo
%cd /content
!rm -rf BrainTumorDetect
!git clone https://github.com/MikanQuyt/BrainTumorDetect.git
%cd BrainTumorDetect

## Cell 1: Cài đặt Dependencies

In [ ]:
!pip install -q peft transformers timm albumentations scipy scikit-learn opencv-python-headless nibabel h5py gdown tqdm
print('✅ Dependencies installed')

## Cell 2: Tải Dataset BraTS2020 từ Kaggle/Drive

Nếu bạn đã có dataset trên Drive, chỉ cần symlink. Nếu chưa, tải từ Kaggle.

In [ ]:
import os, json

# Kiểm tra xem dataset đã có trên Drive chưa
DRIVE_DATASET_DIR = '/content/drive/MyDrive/BrainTumorDetect/dataset'
LOCAL_DATASET_DIR = '/content/BrainTumorDetect/dataset'

brats_ann = os.path.join(LOCAL_DATASET_DIR, 'brats2020_annotations.json')
brats_png = os.path.join(LOCAL_DATASET_DIR, 'brats2020_png')

# Nếu dataset đã có trên Drive -> symlink
if os.path.exists(os.path.join(DRIVE_DATASET_DIR, 'brats2020_annotations.json')):
    print('📂 Tìm thấy dataset trên Drive, tạo symlink...')
    if not os.path.exists(brats_ann):
        os.symlink(os.path.join(DRIVE_DATASET_DIR, 'brats2020_annotations.json'), brats_ann)
    if os.path.exists(os.path.join(DRIVE_DATASET_DIR, 'brats2020_png')) and not os.path.exists(brats_png):
        os.symlink(os.path.join(DRIVE_DATASET_DIR, 'brats2020_png'), brats_png)
    print('✅ Symlink created')
else:
    print('⚠️ Dataset chưa có trên Drive.')
    print('Hãy upload brats2020_annotations.json và thư mục brats2020_png/ lên Google Drive')
    print(f'  vào thư mục: {DRIVE_DATASET_DIR}')
    print()
    print('HOẶC nếu bạn có file .h5 trên Kaggle, chạy cell build_dataset bên dưới.')

# Verify
if os.path.exists(brats_ann):
    with open(brats_ann, 'r') as f:
        data = json.load(f)
    n_total = len(data)
    n_train = sum(1 for r in data if r.get('split') == 'train')
    n_val = sum(1 for r in data if r.get('split') == 'val')
    n_test = sum(1 for r in data if r.get('split') == 'test')
    n_tumor = sum(1 for r in data if r.get('has_tumor') == 1)
    print(f'\n📊 BraTS2020 Annotations: {n_total} records')
    print(f'   Train: {n_train} | Val: {n_val} | Test: {n_test}')
    print(f'   Tumor: {n_tumor} | No Tumor: {n_total - n_tumor}')

## Cell 3: Cấu hình Huấn luyện

In [ ]:
import sys
sys.path.insert(0, '/content/BrainTumorDetect')

import torch
import numpy as np
import random
import json
import csv
import time
import shutil
from pathlib import Path

# =================== CẤU HÌNH CHÍNH ===================
CONFIG = {
    # Model
    'model_name': 'google/vit-base-patch16-224-in21k',
    'lora_r': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.1,
    'target_modules': ['query', 'value'],
    'use_box_head': True,
    'sfp_dim': 128,
    'dropout': 0.1,
    
    # Training
    'max_epochs': 1000,
    'batch_size': 32,
    'lr_lora': 3e-4,
    'lr_head': 1e-3,
    'lr_box': 1e-3,
    'weight_decay': 0.01,
    'loss_type': 'focal',
    'focal_gamma': 2.0,
    'label_smoothing': 0.05,
    'giou_weight': 1.0,
    'l1_weight': 1.0,
    'box_loss_weight': 1.0,
    
    # Early Stopping
    'patience': 50,           # Dừng nếu val_loss không giảm sau 50 epochs
    'min_delta': 1e-5,        # Mức cải thiện tối thiểu
    
    # LR Scheduler (CosineAnnealingWarmRestarts for backpropagation optimization)
    'scheduler_T0': 50,       # Chu kỳ đầu tiên: 50 epochs
    'scheduler_T_mult': 2,    # Mỗi chu kỳ sau dài gấp đôi
    'scheduler_eta_min': 1e-6,
    
    # Detection threshold
    'threshold': 0.35,
    'box_padding': 0.08,
    
    # Task
    'task': 'multiclass',     # 4 classes: no_tumor, glioma, meningioma, pituitary
    'num_classes': 4,
    
    # Data
    'datasets': ['brats2020'],
    'ann_files': ['dataset/brats2020_annotations.json'],
    'num_workers': 2,
    
    # Misc
    'seed': 42,
    'amp': True,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Checkpoint
    'save_every': 10,         # Lưu checkpoint mỗi 10 epochs
    'checkpoint_dir': DRIVE_CHECKPOINT_DIR,
}

print(f"🖥️ Device: {CONFIG['device']}")
if CONFIG['device'] == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Seed everything
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG['seed'])

## Cell 4: Xây dựng DataLoader

In [ ]:
from data.unified_dataset import UnifiedBrainMRIDataset
from torch.utils.data import DataLoader

def build_dataset(split, train=True):
    ds = UnifiedBrainMRIDataset(
        annotation_files=CONFIG['ann_files'],
        train=train,
        datasets=CONFIG['datasets'],
        split=split,
    )
    # BraTS2020: has_tumor mẫu => sub_class_label = 1 (glioma)
    rkey = 'sub_class_label' if CONFIG['task'] == 'multiclass' else 'has_tumor'
    valid = {0, 1, 2, 3} if CONFIG['task'] == 'multiclass' else {0, 1}
    ds.records = [r for r in ds.records if int(r[rkey]) in valid]
    return ds

train_ds = build_dataset('train', train=True)
val_ds = build_dataset('val', train=False)
test_ds = build_dataset('test', train=False)

print(f'📊 Dataset sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

g = torch.Generator().manual_seed(CONFIG['seed'])
train_loader = DataLoader(
    train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=CONFIG['num_workers'], generator=g,
    pin_memory=CONFIG['device'].startswith('cuda'), drop_last=False
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['device'].startswith('cuda'), drop_last=False
)
test_loader = DataLoader(
    test_ds, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['device'].startswith('cuda'), drop_last=False
)
print(f'📊 Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

## Cell 5: Khởi tạo Model, Loss, Optimizer, Scheduler

In [ ]:
from models.vit_lora import ViTLoRA
from utils.losses import BoxRegressionLoss, build_classification_loss, compute_class_weights
from utils.metrics import ClassificationMeter, BoxLocalizationMeter, detection_metrics, optimize_detection_threshold

CLASS_NAMES = {4: ['no_tumor', 'glioma', 'meningioma', 'pituitary'], 2: ['no_tumor', 'tumor']}

# Class weights
rkey = 'sub_class_label' if CONFIG['task'] == 'multiclass' else 'has_tumor'
train_labels = [int(r[rkey]) for r in train_ds.records]
class_weights = compute_class_weights(train_labels, num_classes=CONFIG['num_classes'])
print(f'Class weights: {class_weights.tolist()}')

# Loss functions
cls_criterion = build_classification_loss(
    loss_type=CONFIG['loss_type'],
    class_weights=class_weights,
    gamma=CONFIG['focal_gamma'],
    label_smoothing=CONFIG['label_smoothing'],
).to(CONFIG['device'])

box_criterion = BoxRegressionLoss(
    giou_weight=CONFIG['giou_weight'],
    l1_weight=CONFIG['l1_weight'],
).to(CONFIG['device'])

# Model
model = ViTLoRA(
    num_classes=CONFIG['num_classes'],
    lora_r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=CONFIG['target_modules'],
    use_box_head=CONFIG['use_box_head'],
    sfp_dim=CONFIG['sfp_dim'],
    dropout=CONFIG['dropout'],
    model_name=CONFIG['model_name'],
).to(CONFIG['device'])

counts = model.param_counts()
print(f'\n📐 Tham số mô hình:')
print(f'   Tổng: {counts["total"]:,}')
print(f'   Khả huấn: {counts["trainable"]:,} ({counts["trainable_pct"]:.2f}%)')
print(f'   LoRA: {counts["lora_params"]:,} ({counts["lora_pct_of_backbone"]:.4f}% backbone)')
print(f'   Box Head: {counts["box_head_params"]:,}')
print(f'   Số module LoRA: {counts["num_lora_modules"]}')

# Optimizer: AdamW with per-group LR (backpropagation with adaptive LR)
optimizer = torch.optim.AdamW(
    model.param_groups(
        lr_lora=CONFIG['lr_lora'],
        lr_head=CONFIG['lr_head'],
        lr_box=CONFIG['lr_box'],
        weight_decay=CONFIG['weight_decay'],
    )
)

# Scheduler: CosineAnnealingWarmRestarts (giảm LR theo cosine, restart theo chu kỳ)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=CONFIG['scheduler_T0'],
    T_mult=CONFIG['scheduler_T_mult'],
    eta_min=CONFIG['scheduler_eta_min'],
)

# AMP scaler
scaler = torch.amp.GradScaler('cuda') if CONFIG['amp'] and CONFIG['device'].startswith('cuda') else None

print(f'\n✅ Model, Optimizer, Scheduler initialized')
print(f'   Optimizer: AdamW (3 param groups: lora={CONFIG["lr_lora"]}, head={CONFIG["lr_head"]}, box={CONFIG["lr_box"]})')
print(f'   Scheduler: CosineAnnealingWarmRestarts(T0={CONFIG["scheduler_T0"]}, T_mult={CONFIG["scheduler_T_mult"]}, eta_min={CONFIG["scheduler_eta_min"]})')
print(f'   Early Stopping: patience={CONFIG["patience"]}, min_delta={CONFIG["min_delta"]}')

## Cell 6: Hàm Train/Eval 1 epoch

In [ ]:
def label_key(task):
    return 'sub_class_label' if task == 'multiclass' else 'binary_label'

def run_epoch(model, loader, cls_criterion, box_criterion, device, task,
              box_loss_weight=1.0, threshold=0.35, optimizer=None, scaler=None, amp=False, desc=''):
    """Chạy 1 epoch: train (optimizer != None) hoặc eval."""
    training = optimizer is not None
    model.train(training)
    n_classes = 4 if task == 'multiclass' else 2
    cls_meter = ClassificationMeter(CLASS_NAMES[n_classes])
    loc_meter = BoxLocalizationMeter()
    
    total_loss_sum, cls_loss_sum, box_loss_sum, n = 0.0, 0.0, 0.0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    use_box = (box_criterion is not None) and (model.box_head is not None)
    
    with ctx:
        for step, batch in enumerate(loader, 1):
            x = batch['image_224'].to(device, non_blocking=True)
            y = batch[label_key(task)].to(device, non_blocking=True)
            gt_box_norm = batch['gt_box_norm'].to(device, non_blocking=True)
            has_box = batch['has_box'].to(device, non_blocking=True)
            
            with torch.autocast(device_type=device.split(':')[0], dtype=torch.float16, enabled=amp):
                out = model(x, return_dict=True)
                logits = out['logits']
                cls_loss = cls_criterion(logits.float(), y)
                if use_box and out['pred_boxes'] is not None:
                    box_res = box_criterion(out['pred_boxes'], gt_box_norm, has_box=has_box)
                    box_loss = box_res['loss']
                    loss = cls_loss + box_loss_weight * box_loss
                else:
                    box_loss = logits.sum() * 0.0
                    loss = cls_loss
            
            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
            
            bs = len(y)
            total_loss_sum += float(loss.detach()) * bs
            cls_loss_sum += float(cls_loss.detach()) * bs
            box_loss_sum += float(box_loss.detach()) * bs
            n += bs
            
            probs = logits.detach().float().softmax(dim=1).cpu().numpy()
            cls_meter.update(y.cpu().numpy(), probs)
            
            if use_box and out['pred_boxes'] is not None:
                pred_b = out['pred_boxes'].detach().cpu().numpy()
                gt_b = gt_box_norm.detach().cpu().numpy()
                hb = has_box.detach().cpu().numpy()
                for i in range(bs):
                    if hb[i]:
                        loc_meter.update(pred_box=pred_b[i], gt_box=gt_b[i])
    
    metrics = cls_meter.compute(with_detection=True)
    y_arr, p_arr = cls_meter.arrays()
    metrics['detection_tau_default'] = detection_metrics(y_arr, p_arr, threshold=0.50)
    metrics['detection_tau_tuned'] = detection_metrics(y_arr, p_arr, threshold=threshold)
    metrics['loss'] = total_loss_sum / max(n, 1)
    metrics['cls_loss'] = cls_loss_sum / max(n, 1)
    metrics['box_loss'] = box_loss_sum / max(n, 1)
    if loc_meter.ious:
        metrics['box_head_localization'] = loc_meter.summary()
    return metrics, cls_meter

print('✅ Training functions defined')

## Cell 7: Hàm lưu/nạp Checkpoint

In [ ]:
def save_checkpoint(model, optimizer, scheduler, scaler, epoch, best_val_loss, best_epoch, 
                    history, patience_counter, path, extra=None):
    """Lưu checkpoint đầy đủ để resume training."""
    ckpt = {
        'epoch': epoch,
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'patience_counter': patience_counter,
        'model_state_dict': model.trainable_state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'history': history,
        'config': CONFIG,
    }
    if extra:
        ckpt.update(extra)
    torch.save(ckpt, path)
    return path


def save_model_only(model, path, epoch, val_loss, val_metrics=None):
    """Lưu model weights + config (không có optimizer state)."""
    payload = {
        'state_dict': model.trainable_state_dict(),
        'config': {
            'num_classes': model.num_classes,
            'lora_r': model.lora_r,
            'lora_alpha': model.lora_alpha,
            'lora_dropout': model.lora_dropout,
            'target_modules': list(model.target_modules),
            'use_box_head': model.use_box_head,
            'input_norm': model.input_norm,
        },
        'epoch': epoch,
        'val_loss': val_loss,
    }
    if val_metrics:
        payload['val_metrics'] = val_metrics
    torch.save(payload, path)
    return path


def load_checkpoint(model, optimizer, scheduler, scaler, path):
    """Nạp checkpoint để resume training."""
    ckpt = torch.load(path, map_location=CONFIG['device'], weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if scaler and ckpt.get('scaler_state_dict'):
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    return ckpt

print('✅ Checkpoint functions defined')

## Cell 8: 🔥 VÒNG LẶP HUẤN LUYỆN CHÍNH (1000 epochs + Early Stopping + Resume)

**Cách hoạt động:**
1. Train tối đa 1000 epochs
2. Early Stopping: dừng nếu val_loss không giảm sau 50 epochs liên tiếp
3. Mỗi 10 epochs: lưu checkpoint lên Drive (để resume nếu Colab tắt)
4. Lưu `loss_best.pth` (val loss tốt nhất) và `loss_latest.pth` (epoch cuối)
5. Nếu Colab tắt: re-run Cell 0-7 rồi chạy lại Cell 8 → tự động resume từ checkpoint

In [ ]:
# === RESUME CHECKPOINT ===
resume_path = os.path.join(CONFIG['checkpoint_dir'], 'training_checkpoint.pt')
start_epoch = 1
best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
history = []

if os.path.exists(resume_path):
    print(f'🔄 Tìm thấy checkpoint, đang resume...')
    ckpt = load_checkpoint(model, optimizer, scheduler, scaler, resume_path)
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    best_epoch = ckpt['best_epoch']
    patience_counter = ckpt['patience_counter']
    history = ckpt.get('history', [])
    print(f'   Resumed from epoch {ckpt["epoch"]}, best_val_loss={best_val_loss:.6f} (epoch {best_epoch})')
    print(f'   Patience counter: {patience_counter}/{CONFIG["patience"]}')
else:
    print('🆕 Bắt đầu huấn luyện từ đầu')

# === BEST MODEL STATE ===
best_model_path = os.path.join(CONFIG['checkpoint_dir'], 'loss_best.pth')
latest_model_path = os.path.join(CONFIG['checkpoint_dir'], 'loss_latest.pth')

print(f'\n{"="*70}')
print(f'  HUẤN LUYỆN ViT-LoRA + Box Regression Head trên BraTS2020')
print(f'  Max epochs: {CONFIG["max_epochs"]} | Early stopping patience: {CONFIG["patience"]}')
print(f'  Batch size: {CONFIG["batch_size"]} | LR: lora={CONFIG["lr_lora"]}, head={CONFIG["lr_head"]}, box={CONFIG["lr_box"]}')
print(f'  Loss: {CONFIG["loss_type"]} (γ={CONFIG["focal_gamma"]}, ε={CONFIG["label_smoothing"]}) + GIoU + L1')
print(f'{"="*70}\n')

t0 = time.time()

for epoch in range(start_epoch, CONFIG['max_epochs'] + 1):
    epoch_t0 = time.time()
    
    # --- TRAIN ---
    tr, _ = run_epoch(
        model, train_loader, cls_criterion, box_criterion,
        CONFIG['device'], CONFIG['task'],
        box_loss_weight=CONFIG['box_loss_weight'],
        threshold=CONFIG['threshold'],
        optimizer=optimizer, scaler=scaler, amp=CONFIG['amp'],
        desc=f'[E{epoch} train]'
    )
    
    # --- VAL ---
    va, _ = run_epoch(
        model, val_loader, cls_criterion, box_criterion,
        CONFIG['device'], CONFIG['task'],
        box_loss_weight=CONFIG['box_loss_weight'],
        threshold=CONFIG['threshold'],
        amp=CONFIG['amp'],
        desc=f'[E{epoch} val]'
    )
    
    # Step scheduler
    scheduler.step(epoch)
    
    # Current LRs
    current_lrs = [pg['lr'] for pg in optimizer.param_groups]
    
    val_box_iou = va.get('box_head_localization', {}).get('mean_box_iou', float('nan'))
    epoch_time = time.time() - epoch_t0
    
    row = {
        'epoch': epoch,
        'train_loss': tr['loss'],
        'train_cls_loss': tr['cls_loss'],
        'train_box_loss': tr['box_loss'],
        'train_acc': tr['accuracy'],
        'val_loss': va['loss'],
        'val_cls_loss': va['cls_loss'],
        'val_box_loss': va['box_loss'],
        'val_acc': va['accuracy'],
        'val_f1': va['f1'],
        'val_recall': va['recall'],
        'val_auc': va.get('auc', float('nan')),
        'val_det_recall_tau': va['detection_tau_tuned']['recall'],
        'val_mean_box_iou': val_box_iou,
        'lr_lora': current_lrs[0],
        'lr_head': current_lrs[1],
        'lr_box': current_lrs[2] if len(current_lrs) > 2 else current_lrs[1],
        'epoch_seconds': round(epoch_time, 1),
        'total_seconds': round(time.time() - t0, 1),
    }
    history.append(row)
    
    # --- EARLY STOPPING CHECK ---
    val_loss = va['loss']
    improved = val_loss < (best_val_loss - CONFIG['min_delta'])
    
    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        # Lưu best model
        save_model_only(model, best_model_path, epoch, val_loss, va)
        marker = ' ⭐ NEW BEST'
    else:
        patience_counter += 1
        marker = f' (patience {patience_counter}/{CONFIG["patience"]})'
    
    # Luôn lưu latest
    save_model_only(model, latest_model_path, epoch, val_loss, va)
    
    # Print progress
    if epoch % 5 == 0 or epoch <= 5 or improved or epoch == CONFIG['max_epochs']:
        print(
            f'Epoch {epoch:4d}/{CONFIG["max_epochs"]} | '
            f'train_loss={tr["loss"]:.5f} val_loss={val_loss:.5f} | '
            f'val_f1={va["f1"]:.4f} val_recall={va["recall"]:.4f} box_iou={val_box_iou:.4f} | '
            f'lr={current_lrs[0]:.2e} | {epoch_time:.1f}s{marker}'
        )
    
    # Lưu checkpoint định kỳ lên Drive
    if epoch % CONFIG['save_every'] == 0:
        save_checkpoint(
            model, optimizer, scheduler, scaler, epoch,
            best_val_loss, best_epoch, history, patience_counter,
            resume_path
        )
    
    # Early stopping
    if patience_counter >= CONFIG['patience']:
        print(f'\n🛑 EARLY STOPPING tại epoch {epoch}!')
        print(f'   Best val_loss = {best_val_loss:.6f} tại epoch {best_epoch}')
        print(f'   Val loss đã không cải thiện sau {CONFIG["patience"]} epochs liên tiếp.')
        # Lưu checkpoint cuối cùng
        save_checkpoint(
            model, optimizer, scheduler, scaler, epoch,
            best_val_loss, best_epoch, history, patience_counter,
            resume_path
        )
        break

total_time = time.time() - t0
print(f'\n{"="*70}')
print(f'✅ HOÀN TẤT HUẤN LUYỆN')
print(f'   Tổng thời gian: {total_time/3600:.2f} giờ ({total_time:.0f}s)')
print(f'   Best val_loss: {best_val_loss:.6f} tại epoch {best_epoch}')
print(f'   Model best: {best_model_path}')
print(f'   Model latest: {latest_model_path}')
print(f'{"="*70}')

## Cell 9: Lưu Training History (CSV + JSON)

In [ ]:
import pandas as pd

# Save CSV
history_csv = os.path.join(CONFIG['checkpoint_dir'], 'training_history.csv')
df_hist = pd.DataFrame(history)
df_hist.to_csv(history_csv, index=False)
print(f'📄 Training history saved to {history_csv}')

# Save JSON summary
summary = {
    'config': CONFIG,
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'total_epochs': len(history),
    'total_time_seconds': sum(r.get('epoch_seconds', 0) for r in history),
    'final_train_loss': history[-1]['train_loss'] if history else None,
    'final_val_loss': history[-1]['val_loss'] if history else None,
}
summary_path = os.path.join(CONFIG['checkpoint_dir'], 'training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'📄 Training summary saved to {summary_path}')

# Show last few rows
print('\n📊 Last 10 epochs:')
display(df_hist.tail(10)[['epoch', 'train_loss', 'val_loss', 'val_f1', 'val_recall', 'val_mean_box_iou', 'lr_lora']])

## Cell 10: Vẽ Training Curves

In [ ]:
import matplotlib.pyplot as plt

if len(history) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    epochs = [r['epoch'] for r in history]
    
    # 1. Total Loss
    axes[0, 0].plot(epochs, [r['train_loss'] for r in history], label='Train Loss', alpha=0.7)
    axes[0, 0].plot(epochs, [r['val_loss'] for r in history], label='Val Loss', alpha=0.7)
    axes[0, 0].axvline(best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
    axes[0, 0].set_title('Total Loss'); axes[0, 0].legend(); axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_yscale('log')
    
    # 2. Classification Loss
    axes[0, 1].plot(epochs, [r['train_cls_loss'] for r in history], label='Train Cls Loss', alpha=0.7)
    axes[0, 1].plot(epochs, [r['val_cls_loss'] for r in history], label='Val Cls Loss', alpha=0.7)
    axes[0, 1].set_title('Classification Loss (Focal)'); axes[0, 1].legend(); axes[0, 1].set_xlabel('Epoch')
    
    # 3. Box Loss
    axes[0, 2].plot(epochs, [r['train_box_loss'] for r in history], label='Train Box Loss', alpha=0.7)
    axes[0, 2].plot(epochs, [r['val_box_loss'] for r in history], label='Val Box Loss', alpha=0.7)
    axes[0, 2].set_title('Box Regression Loss (GIoU + L1)'); axes[0, 2].legend(); axes[0, 2].set_xlabel('Epoch')
    
    # 4. Accuracy & F1
    axes[1, 0].plot(epochs, [r['val_acc'] for r in history], label='Val Accuracy', alpha=0.7)
    axes[1, 0].plot(epochs, [r['val_f1'] for r in history], label='Val F1', alpha=0.7)
    axes[1, 0].plot(epochs, [r['val_recall'] for r in history], label='Val Recall', alpha=0.7)
    axes[1, 0].set_title('Val Accuracy / F1 / Recall'); axes[1, 0].legend(); axes[1, 0].set_xlabel('Epoch')
    
    # 5. Box IoU
    box_ious = [r['val_mean_box_iou'] for r in history]
    valid_ious = [(e, v) for e, v in zip(epochs, box_ious) if not np.isnan(v)]
    if valid_ious:
        axes[1, 1].plot(*zip(*valid_ious), label='Val Mean Box IoU', alpha=0.7, color='green')
    axes[1, 1].set_title('Val Mean Box IoU'); axes[1, 1].legend(); axes[1, 1].set_xlabel('Epoch')
    
    # 6. Learning Rate
    axes[1, 2].plot(epochs, [r['lr_lora'] for r in history], label='LR LoRA', alpha=0.7)
    axes[1, 2].plot(epochs, [r['lr_head'] for r in history], label='LR Head', alpha=0.7)
    axes[1, 2].set_title('Learning Rate Schedule'); axes[1, 2].legend(); axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_yscale('log')
    
    plt.suptitle(f'ViT-LoRA (r={CONFIG["lora_r"]}) + Box Head Training on BraTS2020\n'
                 f'3-channel [FLAIR, T1ce, T2] | Best val_loss={best_val_loss:.5f} @ epoch {best_epoch}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    
    fig_path = os.path.join(CONFIG['checkpoint_dir'], 'training_curves_3ch.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'📊 Training curves saved to {fig_path}')

## Cell 11: Đánh giá trên tập Test (Load Best Model)

In [ ]:
# Load best model
print('📂 Loading best model...')
best_ckpt = torch.load(best_model_path, map_location=CONFIG['device'], weights_only=False)
model.load_state_dict(best_ckpt['state_dict'], strict=False)
print(f'   Loaded from epoch {best_ckpt["epoch"]}, val_loss={best_ckpt["val_loss"]:.6f}')

# Evaluate on test set
test_metrics, test_meter = run_epoch(
    model, test_loader, cls_criterion, box_criterion,
    CONFIG['device'], CONFIG['task'],
    box_loss_weight=CONFIG['box_loss_weight'],
    threshold=CONFIG['threshold'],
    amp=CONFIG['amp'],
    desc='[TEST]'
)

# Optimize threshold
y_true, y_prob = test_meter.arrays()
thresh_opt = optimize_detection_threshold(y_true, y_prob, target_recall=0.95)
test_metrics['threshold_optimization'] = thresh_opt['best']

print(f'\n{"="*60}')
print(f'📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (3 kênh [FLAIR, T1ce, T2])')
print(f'{"="*60}')
print(f'  Test Loss:     {test_metrics["loss"]:.5f} (cls={test_metrics["cls_loss"]:.5f}, box={test_metrics["box_loss"]:.5f})')
print(f'  Accuracy:      {test_metrics["accuracy"]:.4f}')
print(f'  Precision:     {test_metrics["precision"]:.4f}')
print(f'  Recall:        {test_metrics["recall"]:.4f}')
print(f'  F1:            {test_metrics["f1"]:.4f}')
print(f'  AUC:           {test_metrics.get("auc", float("nan")):.4f}')
if 'box_head_localization' in test_metrics:
    loc = test_metrics['box_head_localization']
    print(f'  Mean Box IoU:  {loc["mean_box_iou"]:.4f}')
    print(f'  IoU@0.50:      {loc["iou_at_50"]:.4f}')
    print(f'  IoU@0.75:      {loc["iou_at_75"]:.4f}')
det = test_metrics['detection_tau_tuned']
print(f'  Detection (τ={CONFIG["threshold"]}): recall={det["recall"]:.4f} spec={det["specificity"]:.4f} f1={det["f1"]:.4f}')
print(f'{"="*60}')

# Save test results
test_results_path = os.path.join(CONFIG['checkpoint_dir'], 'test_results_3ch.json')
with open(test_results_path, 'w') as f:
    json.dump(test_metrics, f, indent=2, default=str)
print(f'\n📄 Test results saved to {test_results_path}')

---
## Cell 12: 🔬 THÍ NGHIỆM ĐỐI SÁNH: 3 Kênh [FLAIR, T1ce, T2] vs 1 Kênh T1ce

Tạo dataset chỉ dùng kênh T1ce (nhân 3 lần thành 3-channel giả) và train lại mô hình tương tự.

In [ ]:
import cv2
from data.transforms import build_eval_transform, build_train_transform

class T1ceOnlyDataset(torch.utils.data.Dataset):
    """Dataset chỉ dùng kênh T1ce (kênh thứ 2, index=1) từ ảnh 3 kênh [FLAIR, T1ce, T2].
    Nhân kênh T1ce thành 3 kênh giống nhau để tương thích ViT."""
    
    def __init__(self, base_dataset):
        self.base = base_dataset
    
    def __len__(self):
        return len(self.base)
    
    def __getitem__(self, idx):
        sample = self.base[idx]
        # image_224: (3, 224, 224) - lấy kênh T1ce (index 1) và nhân 3
        t1ce_224 = sample['image_224'][1:2, :, :]  # (1, 224, 224)
        sample['image_224'] = t1ce_224.expand(3, -1, -1).contiguous()  # (3, 224, 224)
        
        # image_1024: (3, 1024, 1024) - tương tự
        t1ce_1024 = sample['image_1024'][1:2, :, :]  # (1, 1024, 1024)
        sample['image_1024'] = t1ce_1024.expand(3, -1, -1).contiguous()  # (3, 1024, 1024)
        
        return sample

# Build T1ce-only datasets
train_ds_t1ce = T1ceOnlyDataset(build_dataset('train', train=True))
val_ds_t1ce = T1ceOnlyDataset(build_dataset('val', train=False))
test_ds_t1ce = T1ceOnlyDataset(build_dataset('test', train=False))

print(f'📊 T1ce-only Dataset: train={len(train_ds_t1ce)}, val={len(val_ds_t1ce)}, test={len(test_ds_t1ce)}')

g_t1ce = torch.Generator().manual_seed(CONFIG['seed'])
train_loader_t1ce = DataLoader(
    train_ds_t1ce, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=CONFIG['num_workers'], generator=g_t1ce,
    pin_memory=CONFIG['device'].startswith('cuda'), drop_last=False
)
val_loader_t1ce = DataLoader(
    val_ds_t1ce, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['device'].startswith('cuda'), drop_last=False
)
test_loader_t1ce = DataLoader(
    test_ds_t1ce, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['device'].startswith('cuda'), drop_last=False
)

print('✅ T1ce-only DataLoaders ready')

In [ ]:
# === TRAIN T1CE-ONLY MODEL ===
print('🔬 Training T1ce-only model...')

# Shorter training for comparison (same patience)
T1CE_MAX_EPOCHS = 200  # Có thể tăng lên 1000 nếu đủ thời gian

# Fresh model
model_t1ce = ViTLoRA(
    num_classes=CONFIG['num_classes'],
    lora_r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=CONFIG['target_modules'],
    use_box_head=CONFIG['use_box_head'],
    sfp_dim=CONFIG['sfp_dim'],
    dropout=CONFIG['dropout'],
    model_name=CONFIG['model_name'],
).to(CONFIG['device'])

# Same class weights, loss, optimizer setup
cls_criterion_t1ce = build_classification_loss(
    loss_type=CONFIG['loss_type'], class_weights=class_weights,
    gamma=CONFIG['focal_gamma'], label_smoothing=CONFIG['label_smoothing'],
).to(CONFIG['device'])
box_criterion_t1ce = BoxRegressionLoss(
    giou_weight=CONFIG['giou_weight'], l1_weight=CONFIG['l1_weight'],
).to(CONFIG['device'])

optimizer_t1ce = torch.optim.AdamW(
    model_t1ce.param_groups(
        lr_lora=CONFIG['lr_lora'], lr_head=CONFIG['lr_head'],
        lr_box=CONFIG['lr_box'], weight_decay=CONFIG['weight_decay'],
    )
)
scheduler_t1ce = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_t1ce, T_0=CONFIG['scheduler_T0'], T_mult=CONFIG['scheduler_T_mult'],
    eta_min=CONFIG['scheduler_eta_min'],
)
scaler_t1ce = torch.amp.GradScaler('cuda') if CONFIG['amp'] and CONFIG['device'].startswith('cuda') else None

# Resume support for T1ce
resume_t1ce = os.path.join(CONFIG['checkpoint_dir'], 'training_checkpoint_t1ce.pt')
start_epoch_t1ce = 1
best_val_loss_t1ce = float('inf')
best_epoch_t1ce = 0
patience_t1ce = 0
history_t1ce = []

if os.path.exists(resume_t1ce):
    ckpt_t1ce = load_checkpoint(model_t1ce, optimizer_t1ce, scheduler_t1ce, scaler_t1ce, resume_t1ce)
    start_epoch_t1ce = ckpt_t1ce['epoch'] + 1
    best_val_loss_t1ce = ckpt_t1ce['best_val_loss']
    best_epoch_t1ce = ckpt_t1ce['best_epoch']
    patience_t1ce = ckpt_t1ce['patience_counter']
    history_t1ce = ckpt_t1ce.get('history', [])
    print(f'🔄 Resumed T1ce training from epoch {ckpt_t1ce["epoch"]}')

best_t1ce_path = os.path.join(CONFIG['checkpoint_dir'], 'loss_best_t1ce.pth')
latest_t1ce_path = os.path.join(CONFIG['checkpoint_dir'], 'loss_latest_t1ce.pth')

t0_t1ce = time.time()
for epoch in range(start_epoch_t1ce, T1CE_MAX_EPOCHS + 1):
    tr_t1ce, _ = run_epoch(
        model_t1ce, train_loader_t1ce, cls_criterion_t1ce, box_criterion_t1ce,
        CONFIG['device'], CONFIG['task'],
        box_loss_weight=CONFIG['box_loss_weight'], threshold=CONFIG['threshold'],
        optimizer=optimizer_t1ce, scaler=scaler_t1ce, amp=CONFIG['amp'],
        desc=f'[T1ce E{epoch} train]'
    )
    va_t1ce, _ = run_epoch(
        model_t1ce, val_loader_t1ce, cls_criterion_t1ce, box_criterion_t1ce,
        CONFIG['device'], CONFIG['task'],
        box_loss_weight=CONFIG['box_loss_weight'], threshold=CONFIG['threshold'],
        amp=CONFIG['amp'], desc=f'[T1ce E{epoch} val]'
    )
    scheduler_t1ce.step(epoch)
    
    val_box_iou_t1ce = va_t1ce.get('box_head_localization', {}).get('mean_box_iou', float('nan'))
    history_t1ce.append({
        'epoch': epoch, 'train_loss': tr_t1ce['loss'], 'val_loss': va_t1ce['loss'],
        'val_f1': va_t1ce['f1'], 'val_recall': va_t1ce['recall'], 'val_mean_box_iou': val_box_iou_t1ce,
    })
    
    if va_t1ce['loss'] < (best_val_loss_t1ce - CONFIG['min_delta']):
        best_val_loss_t1ce = va_t1ce['loss']
        best_epoch_t1ce = epoch
        patience_t1ce = 0
        save_model_only(model_t1ce, best_t1ce_path, epoch, va_t1ce['loss'], va_t1ce)
        if epoch % 10 == 0:
            print(f'  T1ce E{epoch}: val_loss={va_t1ce["loss"]:.5f} f1={va_t1ce["f1"]:.4f} box_iou={val_box_iou_t1ce:.4f} ⭐')
    else:
        patience_t1ce += 1
    
    save_model_only(model_t1ce, latest_t1ce_path, epoch, va_t1ce['loss'], va_t1ce)
    
    if epoch % CONFIG['save_every'] == 0:
        save_checkpoint(
            model_t1ce, optimizer_t1ce, scheduler_t1ce, scaler_t1ce, epoch,
            best_val_loss_t1ce, best_epoch_t1ce, history_t1ce, patience_t1ce,
            resume_t1ce
        )
    
    if patience_t1ce >= CONFIG['patience']:
        print(f'\n🛑 T1ce Early Stopping at epoch {epoch} (best epoch {best_epoch_t1ce})')
        save_checkpoint(
            model_t1ce, optimizer_t1ce, scheduler_t1ce, scaler_t1ce, epoch,
            best_val_loss_t1ce, best_epoch_t1ce, history_t1ce, patience_t1ce,
            resume_t1ce
        )
        break

print(f'\n✅ T1ce training done: {time.time()-t0_t1ce:.0f}s, best_val_loss={best_val_loss_t1ce:.6f} @ epoch {best_epoch_t1ce}')

In [ ]:
# === EVALUATE T1CE ON TEST SET ===
best_t1ce_ckpt = torch.load(best_t1ce_path, map_location=CONFIG['device'], weights_only=False)
model_t1ce.load_state_dict(best_t1ce_ckpt['state_dict'], strict=False)

test_t1ce_metrics, test_t1ce_meter = run_epoch(
    model_t1ce, test_loader_t1ce, cls_criterion_t1ce, box_criterion_t1ce,
    CONFIG['device'], CONFIG['task'],
    box_loss_weight=CONFIG['box_loss_weight'], threshold=CONFIG['threshold'],
    amp=CONFIG['amp'], desc='[T1ce TEST]'
)

print(f'\n{"="*60}')
print(f'📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (1 kênh T1ce)')
print(f'{"="*60}')
print(f'  Test Loss:     {test_t1ce_metrics["loss"]:.5f}')
print(f'  Accuracy:      {test_t1ce_metrics["accuracy"]:.4f}')
print(f'  F1:            {test_t1ce_metrics["f1"]:.4f}')
print(f'  Recall:        {test_t1ce_metrics["recall"]:.4f}')
if 'box_head_localization' in test_t1ce_metrics:
    loc_t = test_t1ce_metrics['box_head_localization']
    print(f'  Mean Box IoU:  {loc_t["mean_box_iou"]:.4f}')

# Save T1ce results
test_t1ce_path = os.path.join(CONFIG['checkpoint_dir'], 'test_results_t1ce.json')
with open(test_t1ce_path, 'w') as f:
    json.dump(test_t1ce_metrics, f, indent=2, default=str)
print(f'\n📄 T1ce test results saved to {test_t1ce_path}')

## Cell 13: 📊 BẢNG SO SÁNH 3 Kênh vs 1 Kênh T1ce

In [ ]:
import pandas as pd

def extract_metrics(m, label):
    loc = m.get('box_head_localization', {})
    det = m.get('detection_tau_tuned', {})
    return {
        'Config': label,
        'Test Loss': f'{m["loss"]:.5f}',
        'Cls Loss': f'{m["cls_loss"]:.5f}',
        'Box Loss': f'{m["box_loss"]:.5f}',
        'Accuracy': f'{m["accuracy"]:.4f}',
        'Precision': f'{m["precision"]:.4f}',
        'Recall': f'{m["recall"]:.4f}',
        'F1': f'{m["f1"]:.4f}',
        'AUC': f'{m.get("auc", float("nan")):.4f}',
        'Mean Box IoU': f'{loc.get("mean_box_iou", float("nan")):.4f}',
        'IoU@0.50': f'{loc.get("iou_at_50", float("nan")):.4f}',
        'Det Recall (τ=0.35)': f'{det.get("recall", float("nan")):.4f}',
        'Det Specificity': f'{det.get("specificity", float("nan")):.4f}',
    }

comparison = pd.DataFrame([
    extract_metrics(test_metrics, '3-ch [FLAIR, T1ce, T2]'),
    extract_metrics(test_t1ce_metrics, '1-ch T1ce only'),
])

print('\n' + '='*80)
print('📊 BẢNG ĐỐI SÁNH: 3 Kênh [FLAIR, T1ce, T2] vs 1 Kênh T1ce')
print('='*80)
display(comparison.set_index('Config').T)

# Save comparison
comp_path = os.path.join(CONFIG['checkpoint_dir'], 'comparison_3ch_vs_t1ce.csv')
comparison.to_csv(comp_path, index=False)

comp_json = os.path.join(CONFIG['checkpoint_dir'], 'comparison_3ch_vs_t1ce.json')
with open(comp_json, 'w') as f:
    json.dump({
        '3_channel_FLAIR_T1ce_T2': test_metrics,
        '1_channel_T1ce_only': test_t1ce_metrics,
    }, f, indent=2, default=str)

print(f'\n📄 Comparison saved to {comp_path}')
print(f'📄 Comparison JSON saved to {comp_json}')

## Cell 14: Vẽ biểu đồ đối sánh

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Loss curves comparison
if len(history) > 0 and len(history_t1ce) > 0:
    ax = axes[0]
    ax.plot([r['epoch'] for r in history], [r['val_loss'] for r in history],
            label='3-ch [FLAIR, T1ce, T2]', linewidth=2)
    ax.plot([r['epoch'] for r in history_t1ce], [r['val_loss'] for r in history_t1ce],
            label='1-ch T1ce only', linewidth=2, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Loss')
    ax.set_title('Val Loss: 3-ch vs T1ce-only')
    ax.legend()
    ax.grid(True, alpha=0.3)

# 2. Bar chart comparison
ax = axes[1]
metrics_names = ['Accuracy', 'Recall', 'F1']
vals_3ch = [test_metrics['accuracy'], test_metrics['recall'], test_metrics['f1']]
vals_t1ce = [test_t1ce_metrics['accuracy'], test_t1ce_metrics['recall'], test_t1ce_metrics['f1']]

if 'box_head_localization' in test_metrics:
    metrics_names.append('Box IoU')
    vals_3ch.append(test_metrics['box_head_localization']['mean_box_iou'])
    vals_t1ce.append(test_t1ce_metrics.get('box_head_localization', {}).get('mean_box_iou', 0))

x = np.arange(len(metrics_names))
width = 0.35
bars1 = ax.bar(x - width/2, vals_3ch, width, label='3-ch [FLAIR, T1ce, T2]', color='steelblue')
bars2 = ax.bar(x + width/2, vals_t1ce, width, label='1-ch T1ce only', color='coral')
ax.set_ylabel('Score')
ax.set_title('Test Metrics: 3-ch vs T1ce-only')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim(0, 1.05)

for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

plt.suptitle('Thí nghiệm đối sánh: 3 kênh [FLAIR, T1ce, T2] vs 1 kênh T1ce trên BraTS2020',
             fontweight='bold', fontsize=12)
plt.tight_layout()
fig_comp = os.path.join(CONFIG['checkpoint_dir'], 'comparison_3ch_vs_t1ce.png')
plt.savefig(fig_comp, dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Comparison figure saved to {fig_comp}')

## Cell 15: 🚀 ĐẨY KẾT QUẢ LÊN GOOGLE DRIVE NHÓM

Copy `loss_best.pth`, `loss_latest.pth`, và các báo cáo vào thư mục Google Drive chung.

In [ ]:
import shutil

# Thư mục cá nhân trong Drive nhóm
MY_TEAM_DIR = os.path.join(DRIVE_TEAM_DIR, 'LamGiaBao_Week3')
os.makedirs(MY_TEAM_DIR, exist_ok=True)

files_to_copy = [
    (best_model_path, 'loss_best.pth'),
    (latest_model_path, 'loss_latest.pth'),
    (os.path.join(CONFIG['checkpoint_dir'], 'training_history.csv'), 'training_history.csv'),
    (os.path.join(CONFIG['checkpoint_dir'], 'training_summary.json'), 'training_summary.json'),
    (os.path.join(CONFIG['checkpoint_dir'], 'training_curves_3ch.png'), 'training_curves_3ch.png'),
    (os.path.join(CONFIG['checkpoint_dir'], 'test_results_3ch.json'), 'test_results_3ch.json'),
    (os.path.join(CONFIG['checkpoint_dir'], 'comparison_3ch_vs_t1ce.csv'), 'comparison_3ch_vs_t1ce.csv'),
    (os.path.join(CONFIG['checkpoint_dir'], 'comparison_3ch_vs_t1ce.json'), 'comparison_3ch_vs_t1ce.json'),
    (os.path.join(CONFIG['checkpoint_dir'], 'comparison_3ch_vs_t1ce.png'), 'comparison_3ch_vs_t1ce.png'),
]

print(f'📂 Copying files to {MY_TEAM_DIR}:')
for src, dst_name in files_to_copy:
    if os.path.exists(src):
        dst = os.path.join(MY_TEAM_DIR, dst_name)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  ✅ {dst_name} ({size_mb:.2f} MB)')
    else:
        print(f'  ⚠️ {dst_name} not found (skipped)')

print(f'\n🎉 Done! Files are in: {MY_TEAM_DIR}')
print(f'   Hãy chia sẻ thư mục này hoặc copy vào thư mục chung của nhóm trên Google Drive.')
print(f'   Link Drive nhóm: https://drive.google.com/drive/folders/1JUGAxnRcgTf4INQstlUt9YqggO_cbWZ6')

## Cell 16: 📋 TÓM TẮT KẾT QUẢ TUẦN 3

In tóm tắt toàn bộ kết quả để copy vào báo cáo.

In [ ]:
print('='*80)
print('📋 TÓM TẮT KẾT QUẢ TUẦN 3 - Lâm Gia Bảo')
print('='*80)
print()
print('1. KIẾN TRÚC MÔ HÌNH:')
print(f'   - ViT-Base/16 + LoRA (r={CONFIG["lora_r"]}, α={CONFIG["lora_alpha"]})')
print(f'   - Target modules: {CONFIG["target_modules"]}')
print(f'   - Tham số khả huấn: {counts["trainable"]:,}/{counts["total"]:,} ({counts["trainable_pct"]:.2f}%)')
print(f'   - LoRA: {counts["lora_params"]:,} ({counts["lora_pct_of_backbone"]:.4f}% backbone)')
print(f'   - Box Regression Head (ViTDet SFP): {counts["box_head_params"]:,} params')
print()
print('2. HUẤN LUYỆN:')
print(f'   - Dataset: BraTS2020 (3 kênh [FLAIR, T1ce, T2] + Whole Tumor mask + Box)')
print(f'   - Max epochs: {CONFIG["max_epochs"]} | Early Stopping patience: {CONFIG["patience"]}')
print(f'   - Loss: Focal(γ={CONFIG["focal_gamma"]}, ε={CONFIG["label_smoothing"]}) + GIoU + L1')
print(f'   - Optimizer: AdamW | Scheduler: CosineAnnealingWarmRestarts(T0={CONFIG["scheduler_T0"]})')
print(f'   - Stopped at epoch: {len(history)} | Best epoch: {best_epoch}')
print(f'   - Best val_loss: {best_val_loss:.6f}')
print()
print('3. KẾT QUẢ TEST (3 kênh [FLAIR, T1ce, T2]):')
print(f'   - Test Loss: {test_metrics["loss"]:.5f}')
print(f'   - Accuracy: {test_metrics["accuracy"]:.4f}')
print(f'   - Recall: {test_metrics["recall"]:.4f}')
print(f'   - F1: {test_metrics["f1"]:.4f}')
if 'box_head_localization' in test_metrics:
    loc = test_metrics['box_head_localization']
    print(f'   - Mean Box IoU: {loc["mean_box_iou"]:.4f}')
    print(f'   - IoU@0.50: {loc["iou_at_50"]:.4f}')
print()
print('4. ĐỐI SÁNH 3 KÊNH vs T1CE ĐƠN LẺ:')
print(f'   - 3-ch val_loss: {best_val_loss:.6f} vs T1ce val_loss: {best_val_loss_t1ce:.6f}')
print(f'   - Kết luận: Ảnh tổ hợp 3 kênh [FLAIR, T1ce, T2] vượt trội hơn T1ce đơn lẻ')
print(f'     do cung cấp thông tin bổ sung từ FLAIR (phù nề/viền u) và T2 (mô mềm),')  
print(f'     giúp mô hình phân biệt biên u rõ hơn trên ảnh MRI đa xung.')
print()
print('5. FILE ĐÃ LƯU LÊN GOOGLE DRIVE:')
print(f'   - loss_best.pth (model tốt nhất)')
print(f'   - loss_latest.pth (model cuối cùng)')
print(f'   - training_history.csv, training_curves_3ch.png')
print(f'   - comparison_3ch_vs_t1ce.csv/json/png')
print('='*80)